# PROBLEM 2: CHARACTER-LEVEL NAME GENERATION USING RNN VARIANTS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import string

## TASK-0: DATASET

In [ ]:
# Load the data

def load_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        names = [line.strip().lower() for line in f if line.strip()]
    return names

names = load_data("TrainingNames.txt")

# Build vocabulary
chars = sorted(list(set(''.join(names))))
chars.append('<EOS>')

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)
print(vocab_size)

28


In [ ]:
# Additional functions
def encode(name):
    return [stoi[ch] for ch in name] + [stoi['<EOS>']]

def decode(indices):
    name = ''
    for i in indices:
        if itos[i] == '<EOS>':
            break
        name += itos[i]
    return name

## TASK-1: MODEL IMPLEMENTATION

In [ ]:
# 1. Vanilla Recurrent Neural Network (RNN)

class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        # Hidden state weights
        self.W_xh = nn.Linear(vocab_size, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True)
        # Output weights
        self.W_xy = nn.Linear(vocab_size, vocab_size, bias=False)
        self.W_hy = nn.Linear(hidden_size, vocab_size, bias=True)

    def forward(self, x, h):
        # Hidden state update
        h = torch.tanh(self.W_xh(x) + self.W_hh(h))
        # Output
        y = self.W_xy(x) + self.W_hy(h)
        return y, h

    def init_hidden(self):
        return torch.zeros(1, self.hidden_size)

In [ ]:
# 2. Bidirectional Long Short-Term Memory (BLSTM)

class BLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=vocab_size,
            hidden_size=hidden_size,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(2 * hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

In [ ]:
# 3. RNN with Basic Attention Mechanism

class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        outputs, hidden = self.rnn(x)

        # Attention weights
        attn_weights = torch.softmax(torch.bmm(outputs, outputs.transpose(1, 2)), dim=-1)

        # Context vector
        context = torch.bmm(attn_weights, outputs)

        out = self.fc(context)
        return out

In [ ]:
# Train RNN

def train_rnn(model, epochs=20, lr=0.002):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0
        for name in names:
            encoded = encode(name)
            hidden = model.init_hidden()
            loss = 0

            for i in range(len(encoded)-1):
                x = torch.zeros(1, vocab_size)
                x[0][encoded[i]] = 1
                y = torch.tensor([encoded[i+1]])

                output, hidden = model(x, hidden)
                loss += loss_fn(output, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

In [ ]:
# Train BLSTM

def train_blstm(model, names, epochs=10, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0

        for name in names:
            encoded = encode(name)

            x = torch.zeros(1, len(encoded)-1, vocab_size)
            y = torch.tensor(encoded[1:])

            for t in range(len(encoded)-1):
                x[0][t][encoded[t]] = 1

            output = model(x)
            output = output.view(-1, vocab_size)

            loss = loss_fn(output, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

In [ ]:
# Train RNN with attention
def train_attention(model, names, epochs=10, lr=0.002):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0

        for name in names:
            encoded = encode(name)

            x = torch.zeros(1, len(encoded)-1, vocab_size)
            y = torch.tensor([encoded[-1]])  # predict final char

            for t in range(len(encoded)-1):
                x[0][t][encoded[t]] = 1

            output = model(x)
            output = output.view(-1, vocab_size)
            y = torch.tensor(encoded[1:])
            loss = loss_fn(output, y)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")


In [ ]:
# Generation functions

def generate_rnn(model, start_char='a', max_len=12):
    model.eval()
    with torch.no_grad():
        x = torch.zeros(1, vocab_size)
        x[0][stoi[start_char]] = 1
        hidden = model.init_hidden()

        name = start_char
        for _ in range(max_len):
            output, hidden = model(x, hidden)
            probs = torch.softmax(output, dim=1)
            idx = torch.multinomial(probs, 1).item()

            if itos[idx] == '<EOS>':
                break

            name += itos[idx]
            x = torch.zeros(1, vocab_size)
            x[0][idx] = 1

        return name

def generate_blstm(model, start_char='a', max_len=12):
  model.eval()

  with torch.no_grad():
      name = start_char

      for _ in range(max_len):
          encoded = [stoi[ch] for ch in name]

          x = torch.zeros(1, len(encoded), vocab_size)
          for t, idx in enumerate(encoded):
              x[0][t][idx] = 1

          output = model(x)  # shape: (1, seq_len, vocab_size)
          last_output = output[0, -1]

          probs = torch.softmax(last_output, dim=0)
          idx = torch.multinomial(probs, 1).item()

          if itos[idx] == '<EOS>':
              break

          name += itos[idx]

      return name

def generate_attention(model, start_char='a', max_len=12):
  model.eval()

  with torch.no_grad():
      name = start_char

      for _ in range(max_len):
          encoded = [stoi[ch] for ch in name]

          x = torch.zeros(1, len(encoded), vocab_size)
          for t, idx in enumerate(encoded):
              x[0][t][idx] = 1

          output = model(x)  # (1, vocab_size)
          last_timestep_output = output[0, -1]  # shape: (vocab_size)

          probs = torch.softmax(last_timestep_output, dim=0)
          idx = torch.multinomial(probs, 1).item()

          if itos[idx] == '<EOS>':
              break

          name += itos[idx]

      return name

In [ ]:
# Metrics

def evaluate(model, generator_func, n=1000):
    generated = [generator_func(model, random.choice(string.ascii_lowercase)) for _ in range(n)]

    training_set = set(names)
    unique_generated = set(generated)

    novelty = sum(1 for name in generated if name not in training_set) / n * 100         #calculating novelty
    diversity = len(unique_generated) / n                  #calculate diversity

    return novelty, diversity, generated

## TASK-2: QUANTITATIVE EVALUATION

In [ ]:
# Training and quantitvative evaluation of rnn
hidden_size = 128

rnn_model = VanillaRNN(vocab_size, hidden_size)
train_rnn(rnn_model, epochs=50)

novelty, diversity, samples = evaluate(rnn_model, generate_rnn)

print("\nRNN Results:")
print("Novelty:", novelty)
print("Diversity:", diversity)
print("Samples:", samples[:10])

Epoch 1, Loss: 24823.76
Epoch 2, Loss: 16123.03
Epoch 3, Loss: 12157.83
Epoch 4, Loss: 10265.81
Epoch 5, Loss: 9063.92
Epoch 6, Loss: 8382.57
Epoch 7, Loss: 7857.42
Epoch 8, Loss: 7563.75
Epoch 9, Loss: 7858.78
Epoch 10, Loss: 7007.43
Epoch 11, Loss: 6912.63
Epoch 12, Loss: 9423.14
Epoch 13, Loss: 6946.43
Epoch 14, Loss: 6717.89
Epoch 15, Loss: 6740.68
Epoch 16, Loss: 6694.54
Epoch 17, Loss: 6834.39
Epoch 18, Loss: 7016.22
Epoch 19, Loss: 6446.64
Epoch 20, Loss: 6717.20
Epoch 21, Loss: 6333.91
Epoch 22, Loss: 6234.99
Epoch 23, Loss: 6257.25
Epoch 24, Loss: 6270.13
Epoch 25, Loss: 6363.35
Epoch 26, Loss: 6175.86
Epoch 27, Loss: 6408.46
Epoch 28, Loss: 6121.73
Epoch 29, Loss: 6338.98
Epoch 30, Loss: 6031.90
Epoch 31, Loss: 6029.23
Epoch 32, Loss: 6476.86
Epoch 33, Loss: 6124.20
Epoch 34, Loss: 5986.51
Epoch 35, Loss: 6144.58
Epoch 36, Loss: 6340.32
Epoch 37, Loss: 7596.52
Epoch 38, Loss: 6258.61
Epoch 39, Loss: 5961.81
Epoch 40, Loss: 5980.93
Epoch 41, Loss: 6025.00
Epoch 42, Loss: 6530.

In [ ]:
# Training and quantitvative evaluation of blstm
hidden_size = 128

blstm_model = BLSTM(vocab_size, hidden_size)
train_blstm(blstm_model, names, epochs=20)

novelty2, diversity2, samples2 = evaluate(blstm_model, generate_blstm)

print("\nRNN Results:")
print("Novelty:", novelty2)
print("Diversity:", diversity2)
print("Samples:", samples2[:10])

Epoch 1, Loss: 1158.98
Epoch 2, Loss: 37.80
Epoch 3, Loss: 7.21
Epoch 4, Loss: 2.60
Epoch 5, Loss: 1.25
Epoch 6, Loss: 0.64
Epoch 7, Loss: 0.35
Epoch 8, Loss: 0.20
Epoch 9, Loss: 0.11
Epoch 10, Loss: 0.07
Epoch 11, Loss: 0.04
Epoch 12, Loss: 0.02
Epoch 13, Loss: 0.01
Epoch 14, Loss: 0.01
Epoch 15, Loss: 0.00
Epoch 16, Loss: 0.00
Epoch 17, Loss: 0.00
Epoch 18, Loss: 0.00
Epoch 19, Loss: 0.00
Epoch 20, Loss: 11.30

RNN Results:
Novelty: 100.0
Diversity: 1.0
Samples: ['orrzz ka', 'hozm vy', 'sshirrsshi', 'kall jew', 'chorrjz ge', 'vyyy cch', 'naxdyy gee', 'xenosshrch', 'wall s', 'ttaxxz j']


In [ ]:
# Training and quantitvative evaluation of RNN with attention
attn_model = AttentionRNN(vocab_size, hidden_size=128)
train_attention(attn_model, names, epochs=50)

novelty3, diversity3, samples3 = evaluate(attn_model, generate_attention)

print("Novelty:", novelty3)
print("Diversity:", diversity3)
print("Samples:", samples3[:10])

Epoch 1, Loss: 2363.25
Epoch 2, Loss: 1554.04
Epoch 3, Loss: 1291.02
Epoch 4, Loss: 1155.01
Epoch 5, Loss: 1078.96
Epoch 6, Loss: 1020.61
Epoch 7, Loss: 980.22
Epoch 8, Loss: 950.67
Epoch 9, Loss: 923.30
Epoch 10, Loss: 907.10
Epoch 11, Loss: 888.86
Epoch 12, Loss: 865.81
Epoch 13, Loss: 858.20
Epoch 14, Loss: 844.57
Epoch 15, Loss: 822.83
Epoch 16, Loss: 817.40
Epoch 17, Loss: 797.02
Epoch 18, Loss: 785.09
Epoch 19, Loss: 763.22
Epoch 20, Loss: 764.07
Epoch 21, Loss: 740.40
Epoch 22, Loss: 724.28
Epoch 23, Loss: 711.84
Epoch 24, Loss: 702.21
Epoch 25, Loss: 684.66
Epoch 26, Loss: 689.96
Epoch 27, Loss: 673.43
Epoch 28, Loss: 661.73
Epoch 29, Loss: 672.53
Epoch 30, Loss: 662.30
Epoch 31, Loss: 657.22
Epoch 32, Loss: 650.49
Epoch 33, Loss: 651.79
Epoch 34, Loss: 651.51
Epoch 35, Loss: 646.82
Epoch 36, Loss: 636.15
Epoch 37, Loss: 644.56
Epoch 38, Loss: 638.44
Epoch 39, Loss: 633.28
Epoch 40, Loss: 626.19
Epoch 41, Loss: 626.85
Epoch 42, Loss: 621.37
Epoch 43, Loss: 627.94
Epoch 44, Loss

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("No of parameters of Vanilla RNN:", count_params(rnn_model))
print("No of parameters of BLSTM:", count_params(blstm_model))
print("No of parameters of RNN with attention:", count_params(attn_model))

No of parameters of Vanilla RNN: 24492
No of parameters of BLSTM: 168988
No of parameters of RNN with attention: 40348


In [ ]:
# Evaluate
rnn_metrics = evaluate(rnn_model, generate_rnn)
blstm_metrics = evaluate(blstm_model, generate_blstm)
attn_metrics = evaluate(attn_model, generate_attention)

print("\nFinal Comparison:")
print("RNN:", rnn_metrics[:2])
print("BLSTM:", blstm_metrics[:2])
print("Attention:", attn_metrics[:2])


Final Comparison:
RNN: (72.39999999999999, 0.81)
BLSTM: (100.0, 0.995)
Attention: (96.39999999999999, 0.931)


## TASK-3: QUALITATIVE ANALYSIS

In [ ]:
def print_results(name, metrics):
    novelty, diversity, samples = metrics
    print(f"\n{name}")
    print(f"Novelty Rate: {novelty:.2f}%")
    print(f"Diversity: {diversity:.2f}")
    print("Sample Names:", samples[:10])

print_results("Vanilla RNN", rnn_metrics)
print_results("BLSTM", blstm_metrics)
print_results("Attention RNN", attn_metrics)


Vanilla RNN
Novelty Rate: 72.40%
Diversity: 0.81
Sample Names: ['umesh deshmuk', 'zoya rao', 'bina reddy', 'madhuri kulka', 'nisha mishra', 'xavier mishra', 'quasar banerj', 'quasar george', 'siddyav ghosh', 'omkar kapoor']

BLSTM
Novelty Rate: 100.00%
Diversity: 0.99
Sample Names: ['rrani kul', 'iddan key', 'oseew je', 'kshranjjye', 'kkupkkrjjj', 'kyyakojccx', 'mmkl kalh', 'felk khr', 'kall kho', 'quryannzfy']

Attention RNN
Novelty Rate: 96.40%
Diversity: 0.93
Sample Names: ['zaaa deshmukh', 'pppaa jain', 'eeeesh pillai', 'tejas iyer', 'ojas das', 'naaabsh sharm', 'naaaa kalloo', 'warrm sebasti', 'nnaaa pillai', 'naaannni abra']
